In [1]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # optional: quiet TF logs

In [2]:
import re, random, string
import pandas as pd

# ----------------------------
# Utilities
# ----------------------------
def seed_all(seed=42):
    random.seed(seed)

def maybe(p):
    return random.random() < p

def split_words(text):
    return re.findall(r"\w+|[^\w\s]", text, re.UNICODE)

def join_tokens(tokens):
    # join tokens with simple spacing rules
    out = []
    for i,t in enumerate(tokens):
        if i == 0:
            out.append(t)
        elif re.match(r"[.,!?;:)\]]", t):
            out.append(t)
        elif re.match(r"[(\[]", t):
            out.append(" " + t)
        else:
            out.append(" " + t)
    return "".join(out).strip()

# ----------------------------
# Attack: Typos (character-level)
# ----------------------------
def attack_typos(text, prob=0.08):
    chars = list(text)
    for i in range(len(chars)):
        if chars[i].isalpha() and maybe(prob):
            r = random.random()
            if r < 0.33 and i+1 < len(chars):        # swap with next
                chars[i], chars[i+1] = chars[i+1], chars[i]
            elif r < 0.66:                           # delete char
                chars[i] = ""
            else:                                     # replace char
                chars[i] = random.choice(string.ascii_lowercase)
    return "".join(chars)

# ----------------------------
# Attack: Abbreviations / slang injection
# ----------------------------
SLANG_MAP = {
    "you": "u", "your": "ur", "are": "r", "to": "2", "for": "4",
    "people": "ppl", "because": "bc", "before": "b4", "please": "pls",
    "though": "tho", "really": "rly", "something": "smth",
}
FILLERS = ["idk", "tbh", "ngl", "fr", "lowkey", "literally", "like"]

def attack_slang_abbrev(text, replace_prob=0.18, filler_prob=0.10):
    tokens = split_words(text)
    new = []
    for tok in tokens:
        low = tok.lower()
        if tok.isalpha() and low in SLANG_MAP and maybe(replace_prob):
            # keep capitalization loosely
            rep = SLANG_MAP[low]
            new.append(rep if tok.islower() else rep.upper())
        else:
            new.append(tok)
        # occasionally inject filler after words
        if tok.isalpha() and maybe(filler_prob):
            new.append(random.choice(FILLERS))
    return join_tokens(new)

# ----------------------------
# Attack: Messy punctuation / inconsistencies
# ----------------------------
def attack_punct_mess(text, prob=0.18):
    # add repeated punctuation, odd spacing, random ellipses
    t = text
    if maybe(prob):
        t = re.sub(r"\.", lambda m: "..." if maybe(0.6) else ".", t)
    if maybe(prob):
        t = re.sub(r"!", lambda m: "!!" if maybe(0.7) else "!", t)
    if maybe(prob):
        t = re.sub(r"\?", lambda m: "??" if maybe(0.7) else "?", t)
    if maybe(prob):
        # weird spacing around punctuation
        t = re.sub(r"\s+([.,!?;:])", r" \1", t)
    if maybe(prob):
        # inconsistent casing in a few words
        words = t.split()
        for i in range(len(words)):
            if maybe(0.06) and len(words[i]) > 3:
                w = words[i]
                mid = len(w)//2
                words[i] = w[:mid].lower() + w[mid:].upper()
        t = " ".join(words)
    return t

# ----------------------------
# Attack: Shortening / fragmentation
# ----------------------------
def attack_shortening(text, keep_ratio=0.65):
    # keep first N tokens, drop the rest; add a fragmenty ending
    tokens = split_words(text)
    if len(tokens) < 8:
        return text
    k = max(6, int(len(tokens) * keep_ratio))
    kept = tokens[:k]
    tail = random.choice(["...", " idk", " lol", " smh", " anyway"])
    return join_tokens(kept) + tail

# ----------------------------
# Attack: "Emotional spike" (intensifiers / emojis)
# ----------------------------
INTENSIFIERS = ["so", "super", "actually", "insanely", "wildly", "crazy"]
EMOJI = ["😡", "😤", "😭", "💀", "🔥", "🤦", "🙄"]

def attack_emotional_spike(text, prob=0.25):
    tokens = split_words(text)
    new = []
    for tok in tokens:
        new.append(tok)
        if tok.isalpha() and maybe(prob*0.12):
            new.append(random.choice(INTENSIFIERS))
        if maybe(prob*0.05):
            new.append(random.choice(EMOJI))
    # also maybe add ALL CAPS burst
    if maybe(prob*0.25):
        words = text.split()
        if words:
            i = random.randrange(len(words))
            words[i] = words[i].upper()
            text = " ".join(words)
    return join_tokens(new)

# ----------------------------
# Attack: Ideological steering (lightweight, safe)
# ----------------------------
# Note: This does NOT invent hate content; it just nudges framing with neutral prefixes.
FRAMING_PREFIX = [
    "Hot take:", "Unpopular opinion:", "Real talk:", "Let’s be honest:",
    "I might be wrong but", "From what I’ve seen,"
]

def attack_ideology_steer(text, prob=0.35):
    if maybe(prob):
        return random.choice(FRAMING_PREFIX) + " " + text
    return text

# ----------------------------
# Compose attacks
# ----------------------------
ATTACKS = {
    "typos": attack_typos,
    "slang": attack_slang_abbrev,
    "punct": attack_punct_mess,
    "shorten": attack_shortening,
    "emotion": attack_emotional_spike,
    "steer": attack_ideology_steer,
}

def apply_attack(text, attack_types, strength=1):
    # strength controls how many attacks are composed
    chosen = random.sample(attack_types, k=min(strength, len(attack_types)))
    t = text
    for name in chosen:
        t = ATTACKS[name](t)
    return t, "+".join(chosen)

def augment_ai_posts(df, text_col="text", label_col="label", event_col="event",
                     adv_ratio=0.6, strength=2, attack_pool=None, seed=42):
    """
    Creates extra rows for AI posts only.
    adv_ratio: fraction of AI posts to augment
    strength: number of attack types composed per augmented example (1–3 recommended)
    """
    seed_all(seed)
    if attack_pool is None:
        attack_pool = list(ATTACKS.keys())

    ai = df[df[label_col] == 1].copy()
    n = int(len(ai) * adv_ratio)
    ai_sample = ai.sample(n=n, random_state=seed) if n > 0 else ai.iloc[0:0]

    adv_rows = []
    for idx, row in ai_sample.iterrows():
        adv_text, attack_name = apply_attack(row[text_col], attack_pool, strength=strength)
        adv_rows.append({
            text_col: adv_text,
            label_col: 1,
            event_col: row.get(event_col, None),
            "is_adv": 1,
            "attack_type": attack_name,
            "source_id": idx,
        })

    adv_df = pd.DataFrame(adv_rows)
    base = df.copy()
    base["is_adv"] = 0
    base["attack_type"] = "none"
    base["source_id"] = base.index

    out = pd.concat([base, adv_df], ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out

In [3]:
import os
import pandas as pd

EVENT_FILES = {
    "Election2024": {
        "human": [
            "election2024_tweets.csv",
            "Elec2024Tel.csv"
        ],
        "ai": [
            "election2024_ai_generated.csv",
            "Election2024_synthetic_posts.csv"
        ]
    },

    "Election2020": {
        "human": ["election2020_tweets.csv"],
        "ai": ["election2020_toxic_synthetic.csv"]
    },
    "CapitolAttack":{
        "human":["Capitol_Attack202_tweets.csv",
                "CapitolTel.csv"],
        "ai":["Capitol_Attack_synthetic_posts.csv",
                "Capitol_Attack_synthetic_posts1.csv",
              "capitol_attack_toxic_synthetic.csv"
              ]
    },
    "Covid":{
        "human":["covid_tweets3.csv"],
        "ai":["covid_toxic_textonly.csv",
            "Covid_synthetic_posts.csv"]
    },
    "Summer2020":{
        "human":["Summer2020_tweets2.csv"],
        "ai":["Summer2020_synthetic_posts.csv"]
    },
    "Utah":{
        "human":["Utah_tweets2.csv"],
        "ai":["utah_shooting_toxic_textonly.csv",
            "Utah_Shooting_2025_Dataset.csv"]
    },
    "Roe":{
        "human":["Roe2022_tweets.csv"],
        "ai":[
            "Roe2022_synthetic_posts_v2.csv"]
    },
    "Midterm":{
        "human":["midterm_tweets224.csv"],
        "ai":["Midterm_synthetic_posts.csv"]
    },
    
}

OUT_DIR = "out"
os.makedirs(OUT_DIR, exist_ok=True)

TEXT_COL_CANDIDATES = ["text","Text","content","body","message","post","tweet"]

def find_text_col(df):
    for c in TEXT_COL_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError("Could not find text column")

def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None

def load_one(path, label, event):
    df = pd.read_csv(path)

    text_col = find_text_col(df)
    df = df.rename(columns={text_col:"text"})

    df["text"] = df["text"].apply(clean_text)
    df = df.dropna(subset=["text"]).copy()

    df["label"] = label
    df["event"] = event

    df = df.drop_duplicates(subset=["text"])
    return df[["text","label","event"]]


all_human = []
all_ai = []

for event, paths in EVENT_FILES.items():

    print(f"\nProcessing event: {event}")

    # --- HUMAN FILES ---
    for p in paths["human"]:
        print("  human:", p)
        all_human.append(load_one(p, 0, event))

    # --- AI FILES ---
    for p in paths["ai"]:
        print("  ai:", p)
        all_ai.append(load_one(p, 1, event))


human_df = pd.concat(all_human, ignore_index=True)
ai_df = pd.concat(all_ai, ignore_index=True)

# Remove duplicates across whole dataset
human_df = human_df.drop_duplicates(subset=["text"])
ai_df = ai_df.drop_duplicates(subset=["text"])

combined = pd.concat([human_df, ai_df], ignore_index=True)
combined = combined.sample(frac=1.0, random_state=42).reset_index(drop=True)

human_df.to_csv("out/human_combined_with_event.csv", index=False)
ai_df.to_csv("out/ai_combined_with_event.csv", index=False)
combined.to_csv("out/combined_with_event.csv", index=False)

print("\nDONE")
print("Total:", len(combined))
print("\nEvents distribution:")
print(combined["event"].value_counts())


Processing event: Election2024
  human: election2024_tweets.csv
  human: Elec2024Tel.csv
  ai: election2024_ai_generated.csv
  ai: Election2024_synthetic_posts.csv

Processing event: Election2020
  human: election2020_tweets.csv
  ai: election2020_toxic_synthetic.csv

Processing event: CapitolAttack
  human: Capitol_Attack202_tweets.csv
  human: CapitolTel.csv
  ai: Capitol_Attack_synthetic_posts.csv
  ai: Capitol_Attack_synthetic_posts1.csv
  ai: capitol_attack_toxic_synthetic.csv

Processing event: Covid
  human: covid_tweets3.csv
  ai: covid_toxic_textonly.csv
  ai: Covid_synthetic_posts.csv

Processing event: Summer2020
  human: Summer2020_tweets2.csv
  ai: Summer2020_synthetic_posts.csv

Processing event: Utah
  human: Utah_tweets2.csv
  ai: utah_shooting_toxic_textonly.csv
  ai: Utah_Shooting_2025_Dataset.csv

Processing event: Roe
  human: Roe2022_tweets.csv
  ai: Roe2022_synthetic_posts_v2.csv

Processing event: Midterm
  human: midterm_tweets224.csv
  ai: Midterm_synthetic_po

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
print("✅ transformers imported without TF")


/home/gg2713/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ transformers imported without TF


In [5]:
import sys, site, importlib.util

print("Python:", sys.executable)
print("User site:", site.getusersitepackages())
print("User site in sys.path?", site.getusersitepackages() in sys.path)

print("find_spec('sentencepiece') =", importlib.util.find_spec("sentencepiece"))

try:
    import sentencepiece
    print("✅ sentencepiece imported:", sentencepiece.__version__)
    print("sentencepiece file:", sentencepiece.__file__)
except Exception as e:
    print("❌ sentencepiece import failed:", e)

print("\nFirst 5 sys.path entries:\n", "\n".join(sys.path[:5]))



Python: /share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/bin/python
User site: /home/gg2713/.local/lib/python3.12/site-packages
User site in sys.path? True
find_spec('sentencepiece') = ModuleSpec(name='sentencepiece', loader=<_frozen_importlib_external.SourceFileLoader object at 0x1554fe4c9460>, origin='/home/gg2713/.local/lib/python3.12/site-packages/sentencepiece/__init__.py', submodule_search_locations=['/home/gg2713/.local/lib/python3.12/site-packages/sentencepiece'])
✅ sentencepiece imported: 0.2.1
sentencepiece file: /home/gg2713/.local/lib/python3.12/site-packages/sentencepiece/__init__.py

First 5 sys.path entries:
 /share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/lib/python312.zip
/share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/lib/python3.12
/share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/lib/python3.12/lib-dynload

/home/gg2713/.local/lib/python3.12/site-packages


In [6]:
import site, sys
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)

import sentencepiece
print("✅ sentencepiece:", sentencepiece.__version__)



✅ sentencepiece: 0.2.1


In [7]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base", use_fast=False)
print("✅ tokenizer loaded")


✅ tokenizer loaded


In [8]:
import os, numpy as np, pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate


# NEW: single combined file with event column

DATA_PATH = "out/combined_with_event.csv"
TEXT_COL  = "text"    
LABEL_COL = "label"  
EVENT_COL = "event"

HOLDOUT_EVENT = "Summer2020"  

MODEL_NAME = "microsoft/deberta-v3-base"
OUT_DIR    = "./deberta_event_holdout"

MAX_LEN = 128
SEED = 42
USE_FP16 = False      


# Load + CLEAN 

df = pd.read_csv(DATA_PATH)

assert {TEXT_COL, LABEL_COL, EVENT_COL}.issubset(df.columns)

# Drop bad rows 
df[LABEL_COL] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df = df.dropna(subset=[TEXT_COL, LABEL_COL, EVENT_COL]).copy()

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL] != ""]
df[LABEL_COL] = df[LABEL_COL].astype(int)
df = df[df[LABEL_COL].isin([0, 1])].copy()

print("Rows after cleaning:", len(df))
print("Events:", df[EVENT_COL].value_counts().head(20))
print("Label counts:", df[LABEL_COL].value_counts())


# Event-held-out split

test_df = df[df[EVENT_COL] == HOLDOUT_EVENT].copy()
trainval_df = df[df[EVENT_COL] != HOLDOUT_EVENT].copy()

if len(test_df) == 0:
    raise ValueError(
        f"No rows found for HOLDOUT_EVENT={HOLDOUT_EVENT}. "
        f"Available events:\n{df[EVENT_COL].value_counts()}"
    )

# Train/val split ONLY from trainval 
train_df, val_df = train_test_split(
    trainval_df,
    test_size=0.1,
    random_state=SEED,
    stratify=trainval_df[LABEL_COL]
)

print("\nHoldout:", HOLDOUT_EVENT)
print("Sizes:", len(train_df), len(val_df), len(test_df))
print("Train label dist:\n", train_df[LABEL_COL].value_counts())
print("Test label dist:\n", test_df[LABEL_COL].value_counts())

# -----------------------------
# Apply adversarial augmentation to TRAIN ONLY (before dropping columns)
# -----------------------------
train_df = augment_ai_posts(
    train_df,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    event_col=EVENT_COL,
    adv_ratio=0.6,
    strength=2,
    seed=SEED
)

print("Train size after augmentation:", len(train_df))
print("Adv examples in train:", int(train_df["is_adv"].sum()))
print("Train label dist after aug:\n", train_df[LABEL_COL].value_counts())

# HuggingFace Dataset
# Trainer expects label column named "labels"

train_df = train_df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: "labels"})
val_df   = val_df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: "labels"})
test_df  = test_df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: "labels"})


ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize(batch):
    return tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)

ds_tok = ds.map(tokenize, batched=True, remove_columns=[TEXT_COL])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

# ---- metrics
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=2,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    warmup_steps=500,
    logging_steps=200,
    save_strategy="epoch",
    eval_strategy="epoch" if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames else "epoch",
    fp16=USE_FP16,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ---- sanity check
batch = next(iter(torch.utils.data.DataLoader(ds_tok["train"], batch_size=8, collate_fn=data_collator)))
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    out = model(**batch)
print("Sanity loss (should be finite):", float(out.loss))

trainer.train()

val_metrics = trainer.evaluate(ds_tok["validation"])
print("\nValidation metrics:", val_metrics)

test_out = trainer.predict(ds_tok["test"])
test_preds = np.argmax(test_out.predictions, axis=-1)
test_labels = ds_tok["test"]["labels"]

print("\nConfusion Matrix:\n", confusion_matrix(test_labels, test_preds))
print("\nClassification Report:\n", classification_report(test_labels, test_preds, digits=4))

trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("✅ Saved to", OUT_DIR)

Rows after cleaning: 672122
Events: event
CapitolAttack    177512
Election2024     170353
Covid            112501
Summer2020        76966
Midterm           47827
Roe               40374
Election2020      36457
Utah              10132
Name: count, dtype: int64
Label counts: label
0    424141
1    247981
Name: count, dtype: int64

Holdout: Summer2020
Sizes: 535640 59516 76966
Train label dist:
 label
0    339457
1    196183
Name: count, dtype: int64
Test label dist:
 label
0    46966
1    30000
Name: count, dtype: int64
Train size after augmentation: 653349
Adv examples in train: 117709
Train label dist after aug:
 label
0    339457
1    313892
Name: count, dtype: int64


Map: 100%|██████████| 76966/76966 [00:16<00:00, 4698.11 examples/s]
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Sanity loss (should be finite): 0.6479004621505737


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000000,0.008358,0.998757,0.996662,0.999954,0.998305
2,0.000000,0.001573,0.999748,0.999404,0.999908,0.999656



Validation metrics: {'eval_loss': 0.0015727466670796275, 'eval_accuracy': 0.9997479669332616, 'eval_precision': 0.999403915814572, 'eval_recall': 0.9999082484631617, 'eval_f1': 0.9996560185291352, 'eval_runtime': 121.5627, 'eval_samples_per_second': 489.591, 'eval_steps_per_second': 15.301, 'epoch': 2.0}

Confusion Matrix:
 [[46935    31]
 [   51 29949]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9989    0.9993    0.9991     46966
           1     0.9990    0.9983    0.9986     30000

    accuracy                         0.9989     76966
   macro avg     0.9989    0.9988    0.9989     76966
weighted avg     0.9989    0.9989    0.9989     76966

✅ Saved to ./deberta_event_holdout
